In [55]:
from dotenv import load_dotenv
from google import genai
from pydantic import BaseModel, EmailStr
from typing import List
import pandas as pd
import os, re, json
import duckdb

In [56]:
load_dotenv()
client = genai.Client(api_key=os.getenv("GEMENI_API_KEY"))

In [57]:
response = client.models.generate_content(
    model="gemini-2.5-flash-lite", contents=
    """Skapa ett svenskt företag med precis 20 anställda.
Krav:
- JSON-objekt med fälten: name (str), employees (lista med 20 objekt enligt schemat).
- För varje employee: first_name, last_name (svenska namn), phone_number i formatet "+46 7x xxx xx xx",
  email (matcha namn), department ∈ {"IT","HR","Marknadsföring","Försäljning"},
  salary (rimlig svensk månadslön i SEK som heltal), title (svensk titel som matchar department).
- Returnera endast rå JSON. Inga kodstängsel, ingen extra text, inga ``` json.
"""
)

In [58]:
print(response.text)

{
  "name": "Svenska Innovationer AB",
  "employees": [
    {
      "first_name": "Anna",
      "last_name": "Andersson",
      "phone_number": "+46 70 123 45 67",
      "email": "anna.andersson@svenskainnovationer.se",
      "department": "IT",
      "salary": 45000,
      "title": "IT-specialist"
    },
    {
      "first_name": "Erik",
      "last_name": "Svensson",
      "phone_number": "+46 73 987 65 43",
      "email": "erik.svensson@svenskainnovationer.se",
      "department": "IT",
      "salary": 48000,
      "title": "Systemadministratör"
    },
    {
      "first_name": "Maria",
      "last_name": "Gustafsson",
      "phone_number": "+46 76 555 12 34",
      "email": "maria.gustafsson@svenskainnovationer.se",
      "department": "HR",
      "salary": 42000,
      "title": "HR-generalist"
    },
    {
      "first_name": "Johan",
      "last_name": "Larsson",
      "phone_number": "+46 72 111 22 33",
      "email": "johan.larsson@svenskainnovationer.se",
      "department": "

In [59]:
class Employee(BaseModel):
    first_name: str
    last_name: str
    phone_number: str
    email: EmailStr
    department: str
    salary: int
    title: str

class Company(BaseModel):
    name: str
    employees: List[Employee]

company = Company.model_validate_json(response.text)
company

Company(name='Svenska Innovationer AB', employees=[Employee(first_name='Anna', last_name='Andersson', phone_number='+46 70 123 45 67', email='anna.andersson@svenskainnovationer.se', department='IT', salary=45000, title='IT-specialist'), Employee(first_name='Erik', last_name='Svensson', phone_number='+46 73 987 65 43', email='erik.svensson@svenskainnovationer.se', department='IT', salary=48000, title='Systemadministratör'), Employee(first_name='Maria', last_name='Gustafsson', phone_number='+46 76 555 12 34', email='maria.gustafsson@svenskainnovationer.se', department='HR', salary=42000, title='HR-generalist'), Employee(first_name='Johan', last_name='Larsson', phone_number='+46 72 111 22 33', email='johan.larsson@svenskainnovationer.se', department='HR', salary=44000, title='Rekryterare'), Employee(first_name='Linda', last_name='Karlsson', phone_number='+46 70 777 88 99', email='linda.karlsson@svenskainnovationer.se', department='Marknadsföring', salary=46000, title='Marknadsföringskoord

In [60]:
# dump the json data to folder "output_data"
os.makedirs("output_data", exist_ok=True)

with open("output_data/company.json", "w", encoding="utf-8") as json_file:
    json.dump(
        company.model_dump(),
        json_file,
        ensure_ascii=False,
        indent=2
    )

In [61]:
# put the data into a pandas 
df = pd.DataFrame([emp.model_dump() for emp in company.employees])
df

,first_name,last_name,phone_number,email,department,salary,title
0,Anna,Andersson,+46 70 123 45 67,anna.andersson@svenskainnovationer.se,IT,45000,IT-specialist
1,Erik,Svensson,+46 73 987 65 43,erik.svensson@svenskainnovationer.se,IT,48000,Systemadministratör
2,Maria,Gustafsson,+46 76 555 12 34,maria.gustafsson@svenskainnovationer.se,HR,42000,HR-generalist
3,Johan,Larsson,+46 72 111 22 33,johan.larsson@svenskainnovationer.se,HR,44000,Rekryterare
4,Linda,Karlsson,+46 70 777 88 99,linda.karlsson@svenskainnovationer.se,Marknadsföring,46000,Marknadsföringskoordinator
5,Daniel,Nilsson,+46 73 222 33 44,daniel.nilsson@svenskainnovationer.se,Marknadsföring,49000,Digital Marknadsförare
6,Emma,Pettersson,+46 76 888 99 00,emma.pettersson@svenskainnovationer.se,Försäljning,50000,Säljare
7,Patrik,Berg,+46 72 555 66 77,patrik.berg@svenskainnovationer.se,Försäljning,52000,Account Manager
8,Sara,Lindgren,+46 70 333 44 55,sara.lindgren@svenskainnovationer.se,IT,50000,Frontend-utvecklare
9,Andreas,Holmberg,+46 73 666 77 88,andreas.holmberg@svenskainnovationer.se,IT,53000,Backend-utvecklare


In [62]:
# write a csv file to output_data
df.to_csv("output_data/employees.csv", index=False, encoding="utf-8-sig")
print("File saved correctly")

File saved correctly


In [ ]:
# load the data into a staging layer and store this into a table called employees
con = duckdb.connect("output_data/commpany.duckdb")
con.execute("CREATE SCHEMA IF NOT EXISTS staging;")

con.execute("""
CREATE OR REPLACE TABLE staging.employees AS
SELECT *
FROM read_csv_auto('output_data/employees.csv', header=True)
""")

print(con.execute("SELECT * FROM staging.employees LIMIT 2").fetch_df())

In [64]:
response_ = client.models.generate_content(
    model="gemini-2.5-flash-lite", contents=
    """ You are generating seed data.

    Return in json format ONLY (no prose, no markdown fences), where each element has:
    - department_name: string (MUST be only one each of exactly these: ["IT","HR","Marknadsföring","Försäljning"])
    - description: string (1–2 Swedish sentences)
    - contact_person: string (Swedish full name)
    - contact_email: string (valid email)
    - contact_phone: string (E.164, +46…)

    Do not add or remove departments. One object per department. Remove ```json and ```
    """
)

In [ ]:
print(response_.text)

In [ ]:
class Department(BaseModel):
    department_name: str
    description: str
    contact_person: str
    contact_email: EmailStr 
    contact_phone: str

validate_dep = Department.model_validate_json(response_.text)
validate_dep 

In [ ]:
df_dep = pd.DataFrame([d.model_dump() for d in validate_dep])

df_dep.to_csv("output_data/departments.csv", index=False, encoding="utf-8-sig")
print(df_dep)

  department_name                                        description  \
0              IT  Ansvarar för företagets tekniska infrastruktur...   
1              HR  Hantera personalfrågor och rekrytering samt pe...   
2  Marknadsföring  Skapar och implementerar strategier för att nå...   
3     Försäljning  Driver företagets försäljning och ansvarar för...   

    contact_person                contact_email contact_phone  
0    Anna Svensson    anna.svensson@example.com  +46701234567  
1   Johan Karlsson   johan.karlsson@example.com  +46709876543  
2  Maria Andersson  maria.andersson@example.com  +46702345678  
3   Erik Johansson   erik.johansson@example.com  +46708765432  


In [ ]:
con = duckdb.connect("output_data/commpany.duckdb")

con.execute("""
CREATE OR REPLACE TABLE staging.departments AS 
SELECT * FROM read_csv_auto('output_data/departments.csv', header=True)
""")

print(con.execute("SELECT * FROM staging.departments").fetchdf())

con.close()